In [1]:
import os
from pathlib import Path
print(os.getcwd())
ROOT = Path(os.environ.get("HOME_PROJ_DIR", Path.cwd().resolve().parents[1]))
os.chdir(ROOT)
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/prob/probing_notebooks
/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [2]:
import numpy as np
import pandas as pd
import kinodata.configuration as cfg
from prob.paths_and_io import get_out_dir, load_out_tensor

In [3]:
# this is unnecassary, because I just need the output dir, but maybe in case I need control from args
def get_ds_load_config(**kwargs):
    # default config values
    defaults = dict(
            gnn_model_type="CGNN-3D",
            split_type="random-k-fold",
            filter_rmsd_max_value=2,
            graph_level=True,
            split_index=None,
            dtype_out=None,  # None means no dtype conversion
            device="cpu",
        )
    
    if kwargs.keys() - defaults.keys() != set():
        raise ValueError(f"Invalid arguments: {kwargs.keys() - defaults.keys()}")

    if "gnn_model_type" in kwargs:
        assert kwargs["gnn_model_type"] in ["CGNN-3D", "CGNN", "DTI"], "Invalid GNN model type"
    if "split_type" in kwargs:
        assert kwargs["split_type"] in ["random-k-fold", "scaffold-k-fold", "pocket-k-fold"], "Invalid split type"
    if "filter_rmsd_max_value" in kwargs:
        assert kwargs["filter_rmsd_max_value"] in set({2, 4, 6, 2.00, 4.00, 6.00, None}), "Invalid RMSD threshold"
    if "split_index" in kwargs:
        assert kwargs["split_index"] in set({0, 1, 2, 3, 4, None}), "Invalid split index"
    if "config_name" in kwargs:
        config_name = kwargs["config_name"]
    else:
        config_name = "prob_ds_load"

    # merge: kwargs overrides defaults
    config_args = {**defaults, **kwargs}
    # Initialize the config with the defaults and kwargs
    output_dir = get_out_dir(config_args["gnn_model_type"],
                                config_args["filter_rmsd_max_value"],
                                config_args["split_type"],
                                split_fold=None)
    
    target_dir = output_dir.parents[2] / "targets"

    cfg.register(config_name, **{**config_args, "output_dir": output_dir, "target_dir": target_dir})
    return cfg.get(config_name)

In [4]:
ds_load_config = get_ds_load_config()  # default config
ds_load_config

Config(gnn_model_type=CGNN-3D, split_type=random-k-fold, filter_rmsd_max_value=2, graph_level=True, split_index=None, dtype_out=None, device=cpu, output_dir=/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold, target_dir=/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/targets)

In [13]:
def load_X_from_pt(
        in_dir: str | Path,
        layer_num: int = 1,
        ) -> np.ndarray:
    """ Load a representation tensor from a .pt file and return it as a Numpy array."""
    file_name = f"layer_{layer_num}.pt"
    X_tensor = load_out_tensor(in_dir, file_name)
    return X_tensor.detach().cpu().numpy()

Test it

In [15]:
X = load_X_from_pt(ds_load_config.output_dir, layer_num=1)
print(X.shape) # (num_samples, num_features)

(41238, 256)


In [5]:
ids = load_out_tensor(ds_load_config.output_dir, "ids.pt")
print(ids.shape)  # (num_samples,)

torch.Size([41238])


In [6]:
ids = ids.detach().cpu().numpy().astype(int)

In [33]:
ids[0]

114941

In [9]:
full_targets = load_out_tensor(ds_load_config.target_dir, "nitrogen_counts.pt")
print(len(full_targets))  # (total_num_samples, num_tasks)

119522


In [17]:
target_df = pd.Series(full_targets, dtype="int64")
print(target_df.head)  # (total_num_samples,)


<bound method NDFrame.head of 0         4
17        4
20        3
44        4
1221      4
         ..
119181    4
119182    4
119524    6
119586    6
119593    7
Length: 119522, dtype: int64>


In [11]:
target_df = target_df[ids]
print(target_df.head)  # (num_samples,)

<bound method NDFrame.head of 114941    7
1299      4
47269     8
60380     5
8534      9
         ..
115598    6
105929    6
112881    4
69316     6
46047     6
Length: 41238, dtype: int64>


In [16]:
targte_np = target_df.to_numpy()
targte_np.shape  # (num_samples,)

(41238,)

In [19]:
target_df = target_df.reindex(ids)  # ensure the order matches ids
print(target_df.head)  # (num_samples,)

<bound method NDFrame.head of 114941    7
1299      4
47269     8
60380     5
8534      9
         ..
115598    6
105929    6
112881    4
69316     6
46047     6
Length: 41238, dtype: int64>


RuntimeError: a Tensor with 41238 elements cannot be converted to Scalar

In [ ]:
def load_y_from_ids(
        in_dir: str | Path,
        target_dir: str | Path,
        ids_file: str = "ids.pt",
        targets_file: str = None,
        ) -> np.ndarray:
    """ Load target values corresponding to the given ids from a targets .pt file."""
    if targets_file is None:
        raise ValueError("targets_file must be provided")
    ids = load_out_tensor(in_dir, ids_file)
    full_targets = load_out_tensor(target_dir, targets_file)
    # for easier indexing
    ids = ids.detach().cpu().numpy().astype(int)
    target_df = pd.Series(full_targets, dtype="int64")
    # slice
    target_df = target_df[ids]
    return target_df.to_numpy()

Now that we have the ouput directory, we can read the corresponding .pt file.

# Model

In [ ]:
# For GridSearchCV for hyperparameter tuning
ridge_grid = {"model__alpha": np.logspace(-6, 6, 13)}
lasso_grid = {"model__alpha": np.logspace(-6, 1, 8)}
enet_grid  = {"model__alpha": np.logspace(-6, 3, 10),
              "model__l1_ratio": [0.1, 0.5, 0.9]}
